# Phase 5 — Advanced Extraction Validation

1. New column count/distribution summaries
2. Example queries using new features
3. Before/after comparison for key fields
4. One-click full re-run

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DB = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data/thyroid_master.duckdb')
con = duckdb.connect(str(DB), read_only=True)
print('Connected:', DB)

## 1) New Column Distributions

In [ ]:
print('=== MUTATION FLAGS (tumor_pathology) ===')
mut_cols = ['braf_mutation_mentioned','ras_mutation_mentioned','ret_mutation_mentioned',
            'tert_mutation_mentioned','ntrk_mutation_mentioned','alk_mutation_mentioned']
for c in mut_cols:
    n = con.execute(f'SELECT SUM(CASE WHEN {c} THEN 1 ELSE 0 END) FROM tumor_pathology').fetchone()[0]
    print(f'  {c:35s} TRUE = {n}')

print('\n=== NUCLEAR MED RAI FLAGS ===')
print(con.execute('SELECT rai_avid_flag, COUNT(*) AS n FROM nuclear_med GROUP BY 1 ORDER BY n DESC').fetchdf().to_string(index=False))

print('\n=== NUCLEAR MED METASTASIS SITES (top 15) ===')
print(con.execute("SELECT metastasis_sites, COUNT(*) AS n FROM nuclear_med WHERE metastasis_sites IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 15").fetchdf().to_string(index=False))

print('\n=== US IMPRESSION FLAGS ===')
us_cols = ['us_suspicious_ln','us_vascular_invasion_suspected','us_substernal_suspected','us_calcification_noted']
for c in us_cols:
    n = con.execute(f'SELECT SUM(CASE WHEN {c} THEN 1 ELSE 0 END) FROM ultrasound_reports').fetchone()[0]
    print(f'  {c:40s} TRUE = {n}')

print('\n=== PARATHYROID ADENOMA (tuned) ===')
n = con.execute('SELECT SUM(CASE WHEN is_parathyroid_adenoma THEN 1 ELSE 0 END) FROM parathyroid').fetchone()[0]
print(f'  is_parathyroid_adenoma TRUE = {n}')

## 2) Example Queries Using New Features

In [ ]:
print('=== PTC with BRAF mutation + vascular invasion ===')
df = con.execute('''
SELECT research_id, variant_standardized, largest_tumor_cm, overall_stage_ajcc8,
       tumor_1_vascular_invasion, braf_mutation_mentioned
FROM advanced_features_view
WHERE has_tumor_pathology AND braf_mutation_mentioned = True
  AND tumor_1_vascular_invasion IS NOT NULL
  AND histology_1_type = 'PTC'
ORDER BY largest_tumor_cm DESC
LIMIT 10
''').fetchdf()
display(df)

print('\n=== RAI-avid nuclear med scans with identified metastasis sites ===')
df2 = con.execute('''
SELECT research_id, rai_avid_flag, metastasis_sites, radiotracer, scantype
FROM nuclear_med
WHERE rai_avid_flag = 'positive' AND metastasis_sites IS NOT NULL
LIMIT 10
''').fetchdf()
display(df2)

print('\n=== Ultrasound reports with suspicious LN ===')
df3 = con.execute('''
SELECT research_id, ultrasound_date, us_suspicious_ln, us_calcification_noted, source_us_impression
FROM ultrasound_reports
WHERE us_suspicious_ln = True
LIMIT 5
''').fetchdf()
display(df3)

## 3) Before / After Comparison

In [ ]:
before = {
    'Mutation flags': 0,
    'RAI avidity flag': 0,
    'Metastasis site extraction': 0,
    'US impression flags': 0,
    'Parathyroid adenoma (old filter)': 288,
}

after = {
    'Mutation flags': con.execute('SELECT SUM(CASE WHEN braf_mutation_mentioned OR ras_mutation_mentioned OR ret_mutation_mentioned OR tert_mutation_mentioned OR ntrk_mutation_mentioned OR alk_mutation_mentioned THEN 1 ELSE 0 END) FROM tumor_pathology').fetchone()[0],
    'RAI avidity flag': con.execute("SELECT COUNT(*) FROM nuclear_med WHERE rai_avid_flag IS NOT NULL").fetchone()[0],
    'Metastasis site extraction': con.execute('SELECT COUNT(*) FROM nuclear_med WHERE metastasis_sites IS NOT NULL').fetchone()[0],
    'US impression flags': con.execute('SELECT COUNT(*) FROM ultrasound_reports WHERE us_suspicious_ln OR us_vascular_invasion_suspected OR us_substernal_suspected OR us_calcification_noted').fetchone()[0],
    'Parathyroid adenoma (old filter)': con.execute('SELECT SUM(CASE WHEN is_parathyroid_adenoma THEN 1 ELSE 0 END) FROM parathyroid').fetchone()[0],
}

comp = pd.DataFrame({'Feature': list(before.keys()), 'Before Phase 5': list(before.values()), 'After Phase 5': list(after.values())})
display(comp)

## 4) Visualization: Mutation + RAI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mutation flag counts
mut_counts = {c.replace('_mutation_mentioned',''): con.execute(f'SELECT SUM(CASE WHEN {c} THEN 1 ELSE 0 END) FROM tumor_pathology').fetchone()[0] for c in mut_cols}
axes[0].barh(list(mut_counts.keys()), list(mut_counts.values()), color=sns.color_palette('Set2'))
axes[0].set_title('Mutation Mentions in Pathology Excerpts')
axes[0].set_xlabel('Count')

# RAI avidity distribution
rai = con.execute('SELECT rai_avid_flag, COUNT(*) AS n FROM nuclear_med GROUP BY 1 ORDER BY n DESC').fetchdf()
axes[1].pie(rai['n'], labels=rai['rai_avid_flag'].fillna('NULL'), autopct='%1.1f%%', startangle=90)
axes[1].set_title('RAI Avidity Distribution (Nuclear Med)')

plt.tight_layout()
plt.show()

## 5) One-Click Full Re-Run

In [ ]:
import subprocess

def run_full_pipeline():
    root = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data')
    py = root / '.venv' / 'bin' / 'python'
    for s in ['01_ingest_all_files.py','02_build_duckdb_full.py',
              '05_histology_qa.py','06_advanced_extraction.py',
              '03_research_views.py','04_publication_exports.py']:
        print(f'\n--- {s} ---')
        r = subprocess.run([str(py), str(root/'scripts'/s)], capture_output=True, text=True)
        tail = r.stdout[-400:] if len(r.stdout) > 400 else r.stdout
        print(tail)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-400:])
            break
    print('\nPipeline complete.')

run_full_pipeline()